# Car vs. Bicycle — Training

Place class folders directly inside `data/car-vs-bicycle/`, or change `dataset_dir`.

In [ ]:
import glob
import cv2 as cv
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from keras.models import Sequential,save_model
from keras.layers import Conv2D,MaxPooling2D,Flatten,Dense
import os
import cv2 as cv

In [ ]:
data=[]
labels=[]

In [ ]:
from pathlib import Path

dataset_dir = Path("data/car-vs-bicycle")

In [ ]:
for item in sorted(dataset_dir.glob("*/*")):
    img = cv.imread(str(item)) if item.is_file() else None
    if img is None:
        continue
    f_img = cv.bilateralFilter(img , d=9 , sigmaColor=75 , sigmaSpace=75)
    std_dev = np.std(f_img)
    print(std_dev)
    if std_dev>=0 and std_dev <= 50 :
        alpha = 1.5
    elif std_dev>50 and std_dev <= 100 :
        alpha = 1.0
    elif std_dev > 100 and std_dev <=150 :
        alpha = 0.75
    elif std_dev > 150 :
        alpha = 0.50
    reso_img = cv.convertScaleAbs(f_img,alpha=alpha,beta=0)
    lab = cv.cvtColor(reso_img,cv.COLOR_RGB2Lab)
    l,a,b = cv.split(lab)
    image = cv.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l=image.apply(l)
    lab=cv.merge((l,a,b))
    n_img = cv.cvtColor(lab,cv.COLOR_Lab2RGB)
    r_img = cv.resize(n_img,(50 , 50))
    img = cv.normalize(r_img,None,0,255,cv.NORM_MINMAX)
    data.append(img)
    label = item.parent.name
    labels.append(label)

In [ ]:
data =np.array(data)
le=LabelEncoder()
labels=le.fit_transform(labels)
labels=to_categorical(labels)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(data,labels,test_size=0.02)

In [ ]:
model = Sequential([

    Conv2D(16, (3,3), activation='relu', padding='same', input_shape=(50,50,3)),
    Conv2D(16, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(512, activation='relu'),
    Dense(512, activation='relu'),
    Dense(2, activation='softmax')
])

In [ ]:
model.compile(optimizer="adamax",loss="categorical_crossentropy",metrics=["accuracy"])

In [ ]:
model.fit(x_train,y_train,batch_size=100,epochs=10,validation_data=(x_test,y_test))

In [ ]:
model.save("CAR-or-bike_MODEL.keras")